# Add Capacity to Stations Features

This notebook:
- Loads `stations_features.csv`
- Loads `dublin_bikes_all.parquet`
- Extracts station capacity
- Checks consistency (capacity per station)
- Saves `stations_features_v2.parquet`


In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

stations_path = PROCESSED_DIR / 'stations_features.parquet'
all_path = PROCESSED_DIR / 'dublin_bikes_all.parquet'
if not all_path.exists():
    all_path = PROCESSED_DIR / 'dublin_bikes_all.csv.gz'

output_path = PROCESSED_DIR / 'stations_features_v3.parquet'

# load station-level engineered features
stations = pd.read_parquet(stations_path)

# load raw dataset
if all_path.suffix == '.parquet':
    raw = pd.read_parquet(all_path)
else:
    raw = pd.read_csv(all_path)

raw['last_reported'] = pd.to_datetime(raw['last_reported'], errors='coerce')
raw = raw.sort_values(['station_id', 'last_reported']).copy()

# get one name per station_id
station_names = (
    raw[['station_id', 'name']]
    .dropna(subset=['name'])
    .drop_duplicates(subset='station_id', keep='last')
)

# merge name into stations
stations_v2 = stations.merge(station_names, on='station_id', how='left')

print(stations_v2.columns)
print(stations_v2.head())

Index(['station_id', 'lat', 'lon', 'distance_center', 'center_index',
       'area_type_str', 'tourism', 'name'],
      dtype='str')
   station_id        lat       lon  distance_center  center_index  \
0           1  53.340919 -6.262501         0.009149             1   
1           2  53.356781 -6.268140         0.010497             2   
2           3  53.351185 -6.269859         0.009658             1   
3           4  53.346886 -6.272977         0.013008             2   
4           5  53.330673 -6.260178         0.019127             2   

  area_type_str  tourism                name  
0        retail     True       CLARENDON ROW  
1   residential     True  BLESSINGTON STREET  
2   residential     True       BOLTON STREET  
3   residential     True        GREEK STREET  
4   residential     True    CHARLEMONT PLACE  


In [2]:
# --- Detect columns ---
station_candidates = ["station_id", "stationid", "number", "id"]
capacity_candidates = ["capacity", "bike_stands", "station_capacity", "total_stands"]

station_col = next(c for c in station_candidates if c in raw.columns)
capacity_col = next(c for c in capacity_candidates if c in raw.columns)

print("Station column:", station_col)
print("Capacity column:", capacity_col)


Station column: station_id
Capacity column: capacity


In [3]:
# --- Sort once ---
raw_sorted = raw.sort_values(["station_id", "last_reported"])

# --- Detect changes ---
raw_sorted["capacity_prev"] = raw_sorted.groupby("station_id")[capacity_col].shift()
raw_sorted["capacity_changed"] = raw_sorted[capacity_col] != raw_sorted["capacity_prev"]

# ignore first row per station
raw_sorted["capacity_changed"] &= raw_sorted["capacity_prev"].notna()

# --- Extract change timestamps (first + last change) ---
change_events = raw_sorted[raw_sorted["capacity_changed"]].copy()

change_summary = (
    change_events
    .groupby("station_id")
    .agg(
        first_capacity_change=("last_reported", "min"),
        last_capacity_change=("last_reported", "max"),
    )
    .reset_index()
)

# --- Latest capacity ---
latest_capacity = (
    raw_sorted
    .dropna(subset=[capacity_col])
    .groupby("station_id")
    .tail(1)[["station_id", capacity_col]]
    .rename(columns={capacity_col: "capacity"})
)

# --- Count unique capacities ---
capacity_counts = (
    raw_sorted
    .groupby("station_id")[capacity_col]
    .nunique()
    .reset_index(name="n_unique_capacity")
)

# --- Merge everything ---
capacity_lookup = (
    latest_capacity
    .merge(capacity_counts, on="station_id")
    .merge(change_summary, on="station_id", how="left")
)

capacity_lookup["capacity_non_unique"] = capacity_lookup["n_unique_capacity"] > 1

# --- Inspect ---
print(capacity_lookup[capacity_lookup["capacity_non_unique"]])

    station_id  capacity  n_unique_capacity first_capacity_change  \
68          70        21                  2   2025-05-07 11:15:00   

   last_capacity_change  capacity_non_unique  
68  2025-05-07 11:15:00                 True  


In [11]:

changes = raw_sorted[raw_sorted["capacity_changed"] & raw_sorted["capacity_prev"].notna()]

print(changes[["station_id", "last_reported", "capacity_prev", "capacity"]]
      .sort_values(["station_id", "last_reported"])
      .head(50))
      

          station_id       last_reported  capacity_prev  capacity
11157790          70 2025-05-07 11:15:00           28.0        21


In [19]:
# --- Merge into stations_features ---
stations_v3 = stations_v2.merge(capacity_lookup, on="station_id", how="left")

print(stations_v3.columns)


Index(['station_id', 'lat', 'lon', 'distance_center', 'center_index',
       'area_type_str', 'tourism', 'name', 'capacity', 'n_unique_capacity',
       'first_capacity_change', 'last_capacity_change', 'capacity_non_unique'],
      dtype='str')


In [20]:
# --- Save output ---
stations_v3.to_parquet(output_path, index=False)
print("Saved:", output_path)


Saved: /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/stations_features_v3.parquet
